# Tutorial: Single-cell Workbench Minimal Example

Audience:
- Researchers or engineers who need a reproducible single-cell pipeline scaffold rather than a notebook-only analysis.

Prerequisites:
- Base environment created from `pyproject.toml`.
- The project checked out locally under the current repository.

Learning goals:
- Generate a tiny 10x-style example dataset.
- Run the end-to-end pipeline from config.
- Inspect the exported `h5ad` / `h5mu`, HTML report, and manifest artifacts.

## Outline

1. Resolve the project root and import the package
2. Generate a runnable minimal example
3. Execute the full pipeline
4. Inspect the resulting artifacts and CLI equivalents

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the singlecell_workbench project root.")

project_root = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(project_root / "src"))

from singlecell_workbench.example_data import create_minimal_example
from singlecell_workbench.pipeline import run_pipeline_from_config

project_root

## Step 1 - Generate a tiny 10x-style example

This writes one MTX directory sample and one H5 sample, plus a runnable YAML config.
The example is intentionally tiny so we can verify the workflow before touching real data.

In [ ]:
example_dir = project_root / "example_data" / "minimal_example"
example_manifest = create_minimal_example(example_dir)
print(json.dumps(example_manifest, indent=2))

## Step 2 - Run the full pipeline

This executes ingest, schema validation, QC, annotation, statistics, and reporting from the generated config.

In [ ]:
run_manifest = run_pipeline_from_config(Path(example_manifest["config_path"]))
print(json.dumps({
    "output_dir": run_manifest["output_dir"],
    "final_export": run_manifest["final_export"],
    "report_paths": run_manifest["reports"],
}, indent=2))

## Step 3 - Inspect artifacts

The manifest is the handoff anchor: it records what ran, what was skipped because of optional dependencies, and where every artifact was written.

In [ ]:
manifest_path = Path(run_manifest["run_manifest_path"])
manifest_payload = json.loads(manifest_path.read_text(encoding="utf-8"))
summary = {
    "final_export": manifest_payload["final_export"],
    "schema_fixes": manifest_payload["schema"]["applied_fixes"],
    "qc_status": manifest_payload["qc"],
    "annotation_backend": manifest_payload["annotation"]["selected_backend"],
    "stats_outputs": manifest_payload["stats"],
}
print(json.dumps(summary, indent=2))

## CLI Equivalents

```bash
.venv/bin/scw make-example --output-dir example_data/minimal_example
.venv/bin/scw run --config example_data/minimal_example/run_config.yaml
.venv/bin/scw validate-schema --input-path example_data/minimal_example/data/ctrl_a/filtered_feature_bc_matrix
```

If optional stacks such as `scvi-tools`, `scArches`, `CellTypist`, `SCAR`, or `decoupler` are not installed, the pipeline still completes and records explicit skip reasons in the generated manifests.